In [1]:
import pandas as pd
import openpyxl

In [2]:
file_path = 'suburb_data/bcarr-city-rings-classification-2021.xlsx'
print(f'Reading data from: {file_path}')

# using openpyxl get the sheet name beginning with SA2
sheet_names = openpyxl.load_workbook(file_path, read_only=True).sheetnames
SA2_sheet = [name for name in sheet_names if name.startswith('SA2')][0]

print(f'Sheet names: {sheet_names}')
print(f'SA2 names: {SA2_sheet}')

# read the SA2 sheet into a dataframe
df = pd.read_excel(file_path, sheet_name=SA2_sheet)

# split RING_NAME into three letter city acronym and the rest of the name
df[['city', 'ring_name']] = df['RING_NAME'].str.split(' ', n=1, expand=True)

# keep only MEL rows
df = df[df['city'] == 'MEL'].copy().sort_values('SA2_NAME_2021').reset_index(drop=True)
print(df.shape)

# drop (Vic.) from SA2_NAME_2021
df['SA2_NAME_2021'] = df['SA2_NAME_2021'].str.replace(' \\(Vic.\\)', '', regex=True)

# create a new column which is the count of "-" in SA2_NAME_2021
df['dash_count'] = df['SA2_NAME_2021'].str.count('-')

# split SA2_NAME_2021 using the last "-" as the separator
df[['SA2_NAME_2021', 'suffix']] = df['SA2_NAME_2021'].str.rsplit('-', n=1, expand=True)

replace_list = [' North', ' South', ' East', ' West', ' Central', ' South East', ' South West', ' North East', ' North West']

# replace suffix with None if it is in the replace_list
df['suffix'] = df['suffix'].apply(lambda x: None if x in replace_list else x)
df_suffix = df['suffix'].value_counts().reset_index().rename(columns={'index': 'suffix', 'suffix': 'count'})

# strip all content in brackets from SA2_NAME_2021 and create a new column called SA2_NAME_STRIPPED
df['SA2_NAME_2021_STRIPPED'] = df['SA2_NAME_2021'].str.replace(r'\s*\(.*\)', '', regex=True)

df_col_A = df[['SA2_NAME_2021_STRIPPED', 'ring_name']].copy().rename(columns={'SA2_NAME_2021_STRIPPED': 'Suburb'})
df_col_B = df[['suffix', 'ring_name']].copy().rename(columns={'suffix': 'Suburb'})

df_col_A_B = pd.concat([df_col_A, df_col_B], ignore_index=True)
df_col_A_B['Suburb'] = df_col_A_B['Suburb'].str.strip()

# Hardcode the ring_name for Bundoora and Alphington because they are split between two rings
df_col_A_B.loc[df_col_A_B['Suburb'] == 'Bundoora', 'ring_name'] = 'Outer'
df_col_A_B.loc[df_col_A_B['Suburb'] == 'Alphington', 'ring_name'] = 'Middle'

#print all rows where Suburb is 'Bundoora'
bundoora_rows = df_col_A_B[df_col_A_B['Suburb'] == 'Bundoora']
print('Rows with Suburb as Bundoora:')
print(bundoora_rows)

df_col_A_B = df_col_A_B.dropna(subset=['Suburb']).drop_duplicates().reset_index(drop=True)
print(df_col_A_B['Suburb'].value_counts())
print(df_col_A_B.value_counts())

# Sort by Suburb then print a list of all suburb names
df_col_A_B = df_col_A_B.sort_values('Suburb').reset_index(drop=True)
print('List of suburbs:')
print(df_col_A_B['Suburb'].tolist())    


Reading data from: suburb_data/bcarr-city-rings-classification-2021.xlsx
Sheet names: ['Notes and sources', 'SA2 2021 Concordances', 'Ring Details']
SA2 names: SA2 2021 Concordances
(361, 5)
Rows with Suburb as Bundoora:
      Suburb ring_name
44  Bundoora     Outer
45  Bundoora     Outer
46  Bundoora     Outer
Suburb
Residential         1
Mount Cottrell      1
Cremorne            1
North Warrandyte    1
Windsor             1
                   ..
Altona Meadows      1
Altona              1
Alphington          1
Albert Park         1
Airport West        1
Name: count, Length: 353, dtype: int64
Suburb        ring_name
Yuroke        Outer        1
Werribee      Outer        1
Weir Views    Outer        1
Wattle Glen   Outer        1
Watsonia      Middle       1
                          ..
Alphington    Middle       1
Albion        Middle       1
Albert Park   Inner        1
Airport West  Middle       1
Aberfeldie    Middle       1
Name: count, Length: 353, dtype: int64
List of suburbs:
